# Notebook 02: Data Preprocessing, Imputation & Scaling
**Project**: Maternal-Child Health Profiles Across Countries Through Unsupervised Clustering

### Objectives:
1. Filter countries with excessive indicator missingness ($>35\%$).
2. Justify and apply robust median imputation.
3. Detect statistical outliers and understand their epidemiological meaning.
4. Apply StandardScaler (Z-score standardisation) to equalize feature variance.

In [ ]:
import sys
import os
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data_loader import get_latest_cross_sectional_dataset
from src.preprocessing import HealthDataPreprocessor

### 1. Execute Preprocessing Pipeline

In [ ]:
raw_data = get_latest_cross_sectional_dataset(min_year=2000)
preprocessor = HealthDataPreprocessor(max_missing_country_pct=35.0, imputation_strategy="median")
cleaned, imputed, standardized = preprocessor.fit_transform_pipeline(raw_data, save_artifacts=False)

print(f"Cleaned sample shape: {cleaned.shape}")
print(f"Imputed sample shape: {imputed.shape}")
print(f"Standardized sample shape: {standardized.shape}")

### 2. Outlier Analysis & Verification

In [ ]:
outliers_df = preprocessor.detect_outliers(imputed)
print(outliers_df[["Label", "IQR_Outlier_Count", "ZScore_Outlier_Count", "Min", "Max"]])

### 3. Before and After Standardisation Distributions

In [ ]:
sample_cols = ["skilled_birth_attendance", "under_five_mortality_rate", "child_stunting_prevalence", "improved_sanitation_access"]
fig, axes = plt.subplots(2, len(sample_cols), figsize=(16, 8))
for i, col in enumerate(sample_cols):
    sns.boxplot(y=imputed[col], ax=axes[0, i], color="#aec7e8")
    axes[0, i].set_title(f"Raw: {config.SHORT_TO_LABEL.get(col, col)}", fontsize=10)
    sns.boxplot(y=standardized[col], ax=axes[1, i], color="#2ca02c")
    axes[1, i].set_title(f"Standardized: {config.SHORT_TO_LABEL.get(col, col)}", fontsize=10)
plt.tight_layout()
plt.show()

### 4. Correlation Structure

In [ ]:
corr = preprocessor.compute_correlations(imputed)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Pearson Correlation Heatmap Across Health Indicators")
plt.tight_layout()
plt.show()